In [1]:
import pyodbc
import pandas as pd

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

In [2]:
euro_govs = pd.read_csv('gov_fut.csv')

In [3]:
euro_govs_isin = tuple(euro_govs['isin'].unique())

In [ ]:
# sample: Sep 2024 Bund future (FGBL) on one trading day
sample_isin = 'DE000F0FSQN3'
sample_date = '2024-07-15'

In [ ]:
# Eurex CCP LEIs
eurex_lei = ('529900LN3S50JPU47S06', '529900VIIHQECIS4ET47', '529900UT4DG0LG5R9O07')

In [ ]:
# gov_fut.csv as an inline SQL table, so the final CREATE TABLE can join to it
gov_fut_sql = '\n    UNION ALL '.join(
    f"SELECT '{r.isin}' AS isin, '{r.country}' AS country, '{r.expiration_date}' AS expiration_date"
    for r in euro_govs.itertuples()
)

In [ ]:
# check: exact column names for the trade terms in the REFIT table
cols = pd.read_sql_query("DESCRIBE crp_emir_refit_ecb.emir_refit_ecb_trade_states_deduplicated", cnxn)
cols[cols['name'].str.contains('notional|price|quantity|multiplier', case=False)]

In [ ]:
# step 4: trade terms
query = f"""
WITH gov_fut AS (
    {gov_fut_sql}
),
base AS (
    SELECT
        -- identifiers
        e.tec_ruti,
        e.reference_period,
        -- counterparties
        e.leg1_direction,
        e.leg1_reporting_cpty_id,
        e.leg1_other_cpty_id,
        e.leg1_reporting_cpty_id_country_code_gleif AS rep_country,
        e.leg1_other_cpty_id_country_code_gleif     AS oth_country,
        CASE WHEN e.leg1_reporting_cpty_id IN {eurex_lei} THEN 'CCP' ELSE s_rep.sector END AS rep_sector,
        CASE WHEN e.leg1_other_cpty_id     IN {eurex_lei} THEN 'CCP' ELSE s_oth.sector END AS oth_sector,
        -- product
        e.leg1_product_isin AS product_id,
        g.country           AS product_country,
        g.expiration_date   AS expiration_date,
        -- trade terms
        e.leg1_notional_leg1                 AS notional,
        e.leg1_notional_leg1_currency        AS notional_currency,
        e.leg1_price                         AS price,
        e.leg1_price_currency                AS price_currency,
        e.leg1_total_notional_quantity_leg1  AS quantity,
        e.leg1_price_multiplier              AS price_multiplier
    FROM crp_emir_refit_ecb.emir_refit_ecb_trade_states_deduplicated e
    JOIN gov_fut g
        ON e.leg1_product_isin = g.isin
    LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_rep
        ON e.leg1_reporting_cpty_id = s_rep.lei
    LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_oth
        ON e.leg1_other_cpty_id = s_oth.lei
    WHERE e.reference_period = '{sample_date}'
        AND e.leg1_product_isin = '{sample_isin}'
        AND e.leg1_direction IN ('BYER', 'SLLR')
)
SELECT
    -- identifiers
    tec_ruti,
    reference_period,
    -- counterparties
    CASE WHEN leg1_direction = 'BYER' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS buyer_id,
    CASE WHEN leg1_direction = 'BYER' THEN rep_sector             ELSE oth_sector             END AS buyer_sector,
    CASE WHEN leg1_direction = 'BYER' THEN rep_country            ELSE oth_country            END AS buyer_country,
    CASE WHEN leg1_direction = 'SLLR' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS seller_id,
    CASE WHEN leg1_direction = 'SLLR' THEN rep_sector             ELSE oth_sector             END AS seller_sector,
    CASE WHEN leg1_direction = 'SLLR' THEN rep_country            ELSE oth_country            END AS seller_country,
    -- product
    product_id,
    product_country,
    expiration_date,
    -- trade terms
    notional,
    notional_currency,
    price,
    price_currency,
    quantity,
    price_multiplier
FROM base
"""

df = pd.read_sql_query(query, cnxn)
print(df.shape)
print(df[['notional', 'price', 'quantity', 'price_multiplier']].describe())
df.head()